# MG3 Transformer T1 · Seed 43 · 公榜推理提交

将以下四个文件放在同一目录并上传，以 **mg3_t1_predict.ipynb** 作为提交 Notebook：

| 文件 | 内容 |
|---|---|
| mg3_t1_model.py | 模型、共享清洗、时间掩码、JSON加载、独立训练入口 |
| mg3_t1_predict.py | 平台三参数推理入口与完整截面聚合 |
| mg3_t1_model.json | 真实最佳权重、冻结scaler、结构配置、seed与来源 |
| mg3_t1_predict.ipynb | 公榜导入入口和操作说明 |

默认 Run All 仅导入入口并设置关闭开关，不加载模型JSON、不查询行情、不训练、不调用评估器。
平台随后调用 `main(datasources, start_date, end_date)` 才加载权重，严格返回 **date、instrument、factor** 三列。


In [1]:
from mg3_t1_predict import main
from mg3_t1_model import train_and_save

RUN_MANUAL_TRAIN = False
RUN_FULL_YEAR_QUERY = False
RUN_EVALUATION = False


## 模型与真实权重

- 类型：**MG3_Transformer_T1**，seed=43，参数量 **34,230**。
- 输入：1/5/15/30min，原60根15min网格确定真实回看范围，字段数25；清洗和训练期标准化保持原MG3。
- 多粒度编码、自适应融合、因果CNN不变；时间编码器为单层因果Transformer：d_model=32，4头，FFN64，dropout=0，GELU，pre-norm及最终LayerNorm。
- 固定正弦位置编码保留原历史位置；无效块不作为key/value，全空历史摘要为零；不加入CLS或均值池化。
- 拼接：CNN 24＋Transformer 32＋竞价12＋可用标志1＝69维；个股头和完整截面exclude_self_mean修正保留。
- 本包使用第 **8轮** 最佳检查点，更新 **46,584步**；满8轮结束，最佳点在预算末尾，可能受训练预算限制。
- 2020—2022训练，2023验证选点。2023本地原始RankIC **0.08674506**，Pearson IC **0.08776258**，RankIC IR **0.83374086**；均非官方公榜成绩。
- 权重张量和scaler与研究最佳模型一致，加载时严格核对T1类型和每个张量校验和，拒绝将LSTM权重作为T1加载。

## 平台输入与八个截面

- 行情只读取平台注入的 `datasources["bar1m"]`、`["bar5m"]`、`["bar15m"]`、`["bar30m"]`，缺任何一个均报错，不回退开发行情表。
- 股票池使用注入的 `instruments`；未提供时沿用项目官方池表 `cpt_jyc_2026_instruments`。日度名单仅展开同日成员，日内名单保留各时点成员。
- 八点训练、验证与推理一致：**09:30、10:00、10:30、11:00、11:30、13:30、14:00、14:30**，包含11:30，不含13:00。
- 11:30训练目标按已确认的跨午休口径：13:15与13:30的未来30交易分钟VWAP，结束价为13:30 close；推理不构造或读取标签。
- 历史、竞价输入严格早于决策时刻；回看32个自然日以覆盖原60网格。目标在原标准化空间相加后只逆变换一次。
- 先组装完整时点的股票截面再做均值修正，编码器可以分块，但不能逐股票独立推理后丢失截面上下文。
- 日期直接采用平台实参；纯日期结束日包含当日，有时分秒则保留精确边界。
- 缺失整截面、单截面缺失超过40%、重复键、非有限值或越界均明确报错。

## 可选手动公榜测试

以下是文档示例，不会随 Run All 执行。仅在平台已提供合法表映射和日期后手动运行：

```python
factor_data = main(datasources, start_date, end_date)
assert list(factor_data.columns) == ['date', 'instrument', 'factor']
from bigmodule import M
result = M.jyc_eval._latest(
    factor_data=factor_data, start_date=start_date, end_date=end_date, show=True)
```

## 独立训练、依赖与验收范围

`train_and_save(datasources, config=...)` 是显式独立入口，从初始化训练，main不调用它。
默认2020—2022训练、2023验证；评测日期不会自动用作训练日期。
本包直接使用已完成的seed43权重，不在公榜推理时训练或拟合scaler。

依赖Python 3.10+、NumPy、Pandas、PyTorch及其TransformerEncoder接口；平台提供DAI，手动评价时需要bigmodule。
研究环境为PyTorch 2.10.0.dev20251123+cu128。CPU打包验证不占用正在训练其他seed的GPU。
GPU推理的实际耗时、显存和平台版本兼容性需在平台验证；本地工程验收不等于线上平台验收。
